# Strata Cache Builder — DevZoneX

Prepare the datasets that let a later **T4 ×2** session start Strata in a few
minutes with almost no local disk use.

## What this notebook is for

| Artifact | Dataset | Where it is created |
|---|---|---|
| Compiled runtime (~1.3 GiB) | `strata-runtime-cache` | any session that can install Strata |
| MTP draft layer (~1 GiB final) | `strata-mtp-rt` | section 3c, on CPU |
| Model weights + packs | `strata-model-<family>-<variant>` | section 3c, on CPU |

Then the launcher reads all three in place: local disk holds only the extracted
runtime, and the model is **never copied again**.

## When to enable the GPU — read this twice

| Phase | Accelerator | Why |
|---|---|---|
| 1. Load helpers, restore runtime, probe `strata` | **None (CPU)** | the runtime restore and `--help` need no CUDA |
| 2. Review the plan / probe download | **None (CPU)** | cheap, and tells you the real file list and sizes |
| 3. Download the model, build packs, fetch the draft layer | **None (CPU)** | section 3c calls Strata's own download/preparation helpers directly, so the GPU gate in `setup.py main()` is never entered |
| 4. Publish datasets | **None (CPU)** | upload is file I/O via the Kaggle CLI |
| 5. Every later serving session | **T4 ×2** | use `Strata_Kaggle_Launcher.ipynb`, not this notebook |

`strata install` and `strata launch` both call `setup.py main()`, which stops on a
machine with no NVIDIA or AMD card *before* it downloads anything. That is why
section 3c does not use either command: it imports the installed `setup.py` and
calls its download and preparation functions, which need no GPU.

**Changing the accelerator restarts the Kaggle kernel and wipes `/root`.** Do the
download, the upload, and nothing that changes the accelerator in between.

If a build of Strata does not expose the helpers section 3c uses, the cell says
so explicitly instead of failing silently. In that case, attach **T4 ×2** and run
section 4 to let the normal launch path create the files.


## 0. Configuration

Edit this cell only. The same values must be used by `Strata_Kaggle_Launcher.ipynb`
later, otherwise the datasets will not match the model you launch.


In [ ]:
# ===== USER CONFIGURATION: EDIT THIS CELL =====
KAGGLE_USERNAME = "your-kaggle-username"

# Must match the launcher that will use these datasets.
MODEL_FAMILY = "gyro"
MODEL_VARIANT = "gyro-s"
CONTEXT = "32k"
VISION = "off"
DRAFTING = "auto"                      # "auto"/"mtp" = MTP draft layer; "dflash2" = Qwen3.8-Flash-Next only; "off" = none

MODEL_DATA_DIR = "/root/strata-local"  # everything large lives here
MODEL_RESTORE_MODE = "symlink"         # default: read from attached datasets, do not copy

# One dataset per artifact. The model slug is derived from family/variant.
RUNTIME_DATASET_SLUG = "strata-runtime-cache"
DRAFT_LAYER_DATASET_SLUG = "strata-mtp-rt"
MODEL_DATASET_SLUG = "strata-model-" + f"{MODEL_FAMILY}-{MODEL_VARIANT}".lower()

# Optional external source for the draft layer. Blank = your own
# <KAGGLE_USERNAME>/strata-mtp-rt when attached. The public
# "clothespin/strata-qfn-assets-lite" mirror is opt-in only.
DRAFT_LAYER_DATASET = ""

# Optional raw Strata file bundle to adopt instead of downloading (Dense/PLE
# GGUFs, experts.bin, a profile, or one model's GGUF). It may be a Kaggle
# dataset or a Kaggle model; give "owner/slug" or leave blank.
MODEL_FILES_DATASET = ""

# Upload switches. The model upload additionally requires the license line below.
UPLOAD_RUNTIME_CACHE = True            # ~1.3 GiB
UPLOAD_DRAFT_LAYER = True              # ~1 GiB final (5.2 GB downloaded, packed to mtp/rt)
UPLOAD_MODEL_CACHE = True              # ~55 GiB for Gyro-S; keep True only if disk/time allow
UPLOAD_ADOPTED_MODEL_FILES = False     # True only to copy an attached MODEL_FILES_DATASET into your own dataset
I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE = False
FORCE_UPDATE_RUNTIME_CACHE = False
FORCE_UPDATE_DRAFT_LAYER = False
FORCE_UPDATE_MODEL_CACHE = False

# Probe settings.
PROBE_TIMEOUT_SECONDS = 300

print("Model:", MODEL_FAMILY, MODEL_VARIANT)
print("Model dataset:", f"{KAGGLE_USERNAME}/{MODEL_DATASET_SLUG}")
print("Runtime dataset:", f"{KAGGLE_USERNAME}/{RUNTIME_DATASET_SLUG}")
print("Draft-layer dataset:", DRAFT_LAYER_DATASET or f"{KAGGLE_USERNAME}/{DRAFT_LAYER_DATASET_SLUG}")
print("Data dir:", MODEL_DATA_DIR, "| restore mode:", MODEL_RESTORE_MODE)
print("Uploads runtime/draft/model:", UPLOAD_RUNTIME_CACHE, UPLOAD_DRAFT_LAYER, UPLOAD_MODEL_CACHE)
print("Model license confirmed:", I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE)


## 1. Load helpers — CPU-safe

This clones the public repository, reads the Kaggle token from Secrets, and
reports hardware **without requiring a GPU**. If a `scripts` package import ever
fails in a scratch kernel, the importlib fallback below loads the helper file
directly.


In [ ]:
import importlib.util, json, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/vinodhkumark/devzonex-strata-kaggle.git"
REPO_DIR = Path("/kaggle/working/devzonex-strata-kaggle")
HELPER = REPO_DIR / "scripts" / "strata_kaggle.py"

REQUIRED_HELPERS = (
    "validate_config", "environment_report", "restore_runtime_cache",
    "restore_model_cache", "restore_draft_layer", "cache_status",
    "run_plan", "upload_runtime_cache", "upload_model_cache", "upload_draft_layer",
    "prepare_model_on_cpu", "strata_terminal_python", "find_strata_setup",
)


def helper_is_current() -> bool:
    try:
        text = HELPER.read_text(encoding="utf-8")
    except OSError:
        return False
    return all(("def " + name) in text for name in REQUIRED_HELPERS)


if not helper_is_current():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

spec = importlib.util.spec_from_file_location("strata_kaggle", HELPER)
sk = importlib.util.module_from_spec(spec)
sys.modules["strata_kaggle"] = sk
spec.loader.exec_module(sk)

MISSING = [name for name in REQUIRED_HELPERS if not hasattr(sk, name)]
if MISSING:
    raise RuntimeError("The helper checkout is missing: " + ", ".join(MISSING))

CONFIG = sk.validate_config(
    MODEL_FAMILY, MODEL_VARIANT, CONTEXT, DRAFTING, VISION, MODEL_RESTORE_MODE, "",
)
KAGGLE_ENV = sk.kaggle_env(KAGGLE_USERNAME, secret_name="KAGGLE_API_TOKEN")
HARDWARE = sk.hardware_summary()
print("GPU(s):", HARDWARE["gpu_names"] or "none (CPU session)")
print("Helper:", sk.__file__)


## 2. Restore the runtime and inspect what is attached — CPU-safe

The runtime is the one artifact that must be **extracted locally**: it is an
executable tree (its own Python 3.12, engine binaries, a mutable `terminal/current`),
so a read-only input mount cannot host it.

Attach `strata-runtime-cache` under **Add Input → Your Datasets**. On a returning
session, attaching the model and draft-layer datasets too lets the next cell show
exactly what the launcher will link.


In [ ]:
MODEL_ROOT = Path(MODEL_DATA_DIR)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

ENVIRONMENT = sk.environment_report(require_cuda=False, path=MODEL_DATA_DIR)
RUNTIME_CACHE_RESTORED = sk.restore_runtime_cache(KAGGLE_USERNAME, RUNTIME_DATASET_SLUG)
MODEL_CACHE_RESTORED = sk.restore_model_cache(
    KAGGLE_USERNAME, MODEL_DATASET_SLUG, MODEL_ROOT, workers=8,
    family=MODEL_FAMILY, variant=MODEL_VARIANT, restore_mode=MODEL_RESTORE_MODE,
)

_draft_source = DRAFT_LAYER_DATASET.strip() or (KAGGLE_USERNAME + "/" + DRAFT_LAYER_DATASET_SLUG)
_draft_owner, _, _draft_slug = _draft_source.partition("/")
DRAFT_LAYER_RESTORED = sk.restore_draft_layer(
    _draft_owner.strip(), _draft_slug.strip(), MODEL_ROOT, restore_mode=MODEL_RESTORE_MODE,
)

sk.install_strata_if_missing()
CACHE_STATUS = sk.cache_status(MODEL_ROOT)

print("Runtime cache restored:", RUNTIME_CACHE_RESTORED)
print("Model cache restored:", MODEL_CACHE_RESTORED)
print("Draft layer restored:", DRAFT_LAYER_RESTORED)
print("Restore mode:", MODEL_RESTORE_MODE)
print("Strata executable:", shutil.which("strata") or "(not on PATH yet)")


## 3. Probe Strata — CPU-safe

This is the cell that answers "can the data phase run without a GPU?". It runs
`strata --help`, `strata launch --help`, and the launch plan, then writes the
whole transcript to `/kaggle/working/strata-probe.txt`.

The plan is cheap and runs before any large download. It is the only reliable
source for the model's real file list and sizes.


In [ ]:
STRATA = shutil.which("strata")
if not STRATA:
    raise RuntimeError(
        "strata is not installed. Either attach strata-runtime-cache and rerun the "
        "runtime cell, or accept that this notebook cannot probe a fresh environment."
    )

transcript = []


def record(title, result):
    block = ("### " + title + "\nexit: " + str(result.returncode)
             + "\n--- stdout ---\n" + (result.stdout or "")
             + "\n--- stderr ---\n" + (result.stderr or "") + "\n")
    transcript.append(block)
    print(block)


for title, cmd in (("strata --help", [STRATA, "--help"]),
                   ("strata launch --help", [STRATA, "launch", "--help"])):
    record(title, subprocess.run(cmd, text=True, capture_output=True,
                                 timeout=PROBE_TIMEOUT_SECONDS, check=False))

try:
    PLAN_OUTPUT = sk.run_plan(
        family=MODEL_FAMILY, variant=MODEL_VARIANT, context=CONTEXT, vision=VISION,
        gpu_list="0", data_dir=MODEL_ROOT, drafting=DRAFTING,
    )
    PLAN_OK = True
except Exception as exc:
    PLAN_OUTPUT = "PLAN FAILED: " + str(exc)
    PLAN_OK = False

print(PLAN_OUTPUT)
transcript.append("### plan (ok=" + str(PLAN_OK) + ")\n" + PLAN_OUTPUT + "\n")

PROBE_PATH = Path("/kaggle/working/strata-probe.txt")
PROBE_PATH.write_text("\n".join(transcript), encoding="utf-8")
print("Transcript written to", PROBE_PATH)


## 3b. Read the probe result

`CPU_FETCH_HINTS` are words that would indicate a fetch/prepare step that does not
need CUDA. A hit is a lead, not proof — confirm the subcommand actually downloads
the model files by reading its help output.


In [ ]:
import re

# Separate the recorded blocks so prose in one help page cannot be mistaken for
# a command in another.
BLOCKS = {}
for block in "\n".join(transcript).split("### ")[1:]:
    title, _, body = block.partition("\n")
    BLOCKS[title.strip()] = body

ROOT_HELP = BLOCKS.get("strata --help", "")
LAUNCH_HELP = BLOCKS.get("strata launch --help", "")

# A subcommand line is indented and starts a column of its own.
SUBCOMMAND = re.compile(r"^\s{2,}([a-z][a-z0-9-]+)\s{2,}\S")
SUBCOMMANDS = sorted({
    match.group(1)
    for match in (SUBCOMMAND.match(line) for line in ROOT_HELP.splitlines())
    if match
})
FLAGS = sorted(set(re.findall(r"--[a-z][a-z0-9-]+", LAUNCH_HELP)))

print("Subcommands found under 'strata':", SUBCOMMANDS or "none parsed")
print("Flags found under 'strata launch':", FLAGS or "none parsed")

# Only report a hint when the word is an actual command or flag, not a sentence.
CPU_FETCH_WORDS = ("fetch", "download", "prepare", "prefetch", "pull", "warm")
COMMAND_HITS = [w for w in CPU_FETCH_WORDS
                if any(w == s or w in s.split("-") for s in SUBCOMMANDS)]
FLAG_HITS = [w for w in CPU_FETCH_WORDS if any(w in f for f in FLAGS)]
print("CPU fetch candidates as subcommands:", COMMAND_HITS or "none")
print("CPU fetch candidates as flags:", FLAG_HITS or "none")
print("Plan produced output:", PLAN_OK)

if not PLAN_OK:
    print("\n=> One GPU session is required: the plan itself needs a usable GPU session.")
elif COMMAND_HITS:
    print("\n=> Run '<command> --help' for each candidate above. A real fetcher downloads")
    print("   model files without CUDA; upload them from the same CPU session.")
elif FLAG_HITS:
    print("\n=> A launch flag may limit the run to downloading. Read its help text, then")
    print("   use a download-only launch here on CPU instead of the GPU section.")
else:
    print("\n=> 'download'/'fetch' only appeared in descriptions, not as commands or flags.")
    print("   That is expected: setup.py has no download-only command either.")
    print("   Section 3c therefore calls setup.py's own download and preparation")
    print("   helpers directly, which never enter the GPU-gated main().")

PLAN_TEXT = BLOCKS.get("plan (ok=" + str(PLAN_OK) + ")", PLAN_OUTPUT)
SIZE_HITS = re.findall(r"\b\d+(?:\.\d+)?\s*(?:GiB|GB|MiB|MB)\b", PLAN_TEXT)
print("\nSizes mentioned by the plan:", SIZE_HITS[:20] or "none parsed")
print("Model files mentioned by the plan:",
      sorted(set(re.findall(r"[A-Za-z0-9._/-]+\.(?:gguf|bin)", PLAN_TEXT)))[:20] or "none parsed")
print("\nPaste this whole cell's output plus the 'strata launch --help' block before")
print("trusting any size or filename.")


## 3c. Download and prepare on CPU, without serving it

Run this on the CPU session. It imports the installed `setup.py` and calls the
same helpers its install path uses, in this order:

| Step | Helper | Result |
|---|---|---|
| Download the model | `setup.download()` | one pinned GGUF (58.5 GB), SHA-256 verified, resumable |
| Build the pack | `tools/iq_pack.py` | `packs/gyro-gyro-s/` (index + tokenizer; Gyro reads its experts from the GGUF) |
| Fetch the draft layer | `tools/mtp_fetch.py`, `mtp_pack.py`, `mtp_rt.py` | `mtp/rt/` (~1 GiB final; 5.2 GB downloaded) |

None of these calls `setup.py main()`, so there is no NVIDIA check and no engine
build. Everything is resumable: a finished file is skipped, and a partial
download continues where it stopped.

Budget from the plan: **~63.7 GB** (model 58.5 GB + draft layer 5.2 GB). Gyro's
pack does not copy the quantized experts: they stay in the GGUF and are read at
start, so the pack is the index, the tokenizer and the small float tensors.

`MODEL_DATA_DIR` is `/root/strata-local`, which is session scratch. It is **not**
inside `/kaggle/working`, so the 20 GB output quota does not apply. `/tmp`,
`/root` and `/kaggle/temp` are not separate larger disks; do not download to one
and copy to another, because that briefly needs twice the space.

Kaggle's **writable layer is about 70 GB in total**, and Kaggle staff recommend
staying under ~60 GB. `df` can show the host's multi-terabyte overlay instead;
that number is not the notebook's quota. 63.7 GB is about **59.3 GiB**, which is
what a quota-aware `df` reports. If `shutil.disk_usage(MODEL_ROOT)` shows too
little, change `MODEL_DATA_DIR` to `/kaggle/temp/strata-local`.

A single pass peaks at roughly 63 GiB: the 54.5 GiB GGUF, about 7 GB of MTP
temporaries (5.2 GB of raw tensors, a ~0.9 GB packed GGUF and the ~0.8 GiB
`mtp/rt`), and the runtime. That is at the edge of Kaggle's writable layer, so
**use the two passes below for Gyro-S**. For a smaller model, one pass may fit;
set `CPU_FETCH_DRAFT = False` when the model plus pack is already close to
55 GiB.

After it finishes, `models/`, `packs/` and `mtp/` must contain real bytes, then
run section 5 in this same session — no GPU attached.

### If the GGUF is already in a dataset

A raw bundle that already holds the GGUF (and its `.gguf.done` mark) does not
need downloading. Add it with **Add Input**, then set:

```python
MODEL_VARIANT = "gyro-m"
MODEL_FILES_DATASET = "owner/slug"
CPU_DOWNLOAD_MODEL = False
CPU_BUILD_PACK = True
```

`adopt_model_bundle()` links the files from `/kaggle/input` into `models/`, and
the cell builds `packs/` from them. `UPLOAD_ADOPTED_MODEL_FILES` stays `False`,
so section 5 uploads only the pack; the model stays in its own dataset.

A Kaggle **model** works as well as a dataset. Kaggle mounts it at
`/kaggle/input/models/<owner>/<slug>/gguf/default/1`; give just `owner/slug`
(for example
`buianhtruc/strata-qwen3-8-flash-next-gyro-m-tq2-0`) and the helper finds the
version directory that holds the GGUF.

A multi-shard model (the IQ2/IQ3 sets are two GGUF files) works too: give the
folder and the helper passes the `-00001-of-0000N` shard to `iq_pack.py`, which
reads the other shards beside it.

Gyro-M is **91.97 GB**, larger than Kaggle's whole ~70 GB writable layer. Never
set `CPU_DOWNLOAD_MODEL = True` for it and never use `MODEL_RESTORE_MODE =
"copy"`. The `.done` file is Strata's finish mark; its content does not matter,
but keep it beside the GGUF so a later `setup.download()` skips the file.

### For Gyro-S: two passes

**Pass 1 — the model**

1. Set `CPU_FETCH_DRAFT = False`; keep `CPU_DOWNLOAD_MODEL = True` and
   `CPU_BUILD_PACK = True`. Run this cell, then section 5 with
   `UPLOAD_MODEL_CACHE = True` and `UPLOAD_DRAFT_LAYER = False`.
2. Wait until section 6 says the model dataset is **Ready**. Do not delete
   anything before that.
3. Free the model and the pack, keeping the runtime:

```python
import shutil
for name in ("models", "packs"):
    shutil.rmtree(MODEL_ROOT / name, ignore_errors=True)
```

**Pass 2 — the drafter**

4. Set `CPU_DOWNLOAD_MODEL = False`, `CPU_BUILD_PACK = False` and
   `CPU_FETCH_DRAFT = True`. Run this cell; it does not need the GGUF because
   the MTP tensors come from the original Qwen checkpoint.
5. Run section 5 with `UPLOAD_MODEL_CACHE = False`, `UPLOAD_DRAFT_LAYER = True`
   and `UPLOAD_RUNTIME_CACHE = False` (the runtime is already uploaded).
6. Wait for **Ready**, then remove the MTP download and GGUF, keeping only the
   uploaded runtime files in `mtp/rt`:

```python
import shutil
for path in (MODEL_ROOT / "mtp").iterdir():
    if path.name != "rt":
        shutil.rmtree(path, ignore_errors=True) if path.is_dir() else path.unlink(missing_ok=True)
```

The draft dataset itself is only about 1 GiB; the 5.2 GB figure is the raw MTP
download, not the uploaded layer.


In [ ]:
CPU_PREPARE_MODEL = False   # set True to download and prepare on this CPU session
CPU_DOWNLOAD_MODEL = True   # 58.5 GB for Gyro-S, 92 GB for Gyro-M (SHA-256 verified)
CPU_BUILD_PACK = True       # small: index + tokenizer
CPU_FETCH_DRAFT = True      # 5.2 GB MTP tensors -> mtp/rt

CPU_SUMMARY = {}
MODEL_FILES_PATH = None
if CPU_PREPARE_MODEL:
    if MODEL_FILES_DATASET.strip() and not CPU_DOWNLOAD_MODEL:
        # A raw bundle already holds the GGUF (and usually its .done mark).
        # Symlink it from /kaggle/input so a model larger than Kaggle's ~70 GB
        # writable layer never lands on local disk; build only packs/ here.
        _bundle_owner, _, _bundle_slug = MODEL_FILES_DATASET.partition("/")
        MODEL_FILES_PATH = sk.adopt_model_bundle(
            _bundle_owner.strip() or KAGGLE_USERNAME, _bundle_slug.strip(), MODEL_ROOT,
            MODEL_FAMILY, MODEL_VARIANT, restore_mode=MODEL_RESTORE_MODE,
        )
        print("Using attached model files:", MODEL_FILES_PATH)

    free_gb = shutil.disk_usage(MODEL_ROOT).free / (1024 ** 3)
    print(f"Free space on {MODEL_DATA_DIR}: {free_gb:.1f} GiB")
    if MODEL_FILES_PATH:
        needed_gib = 12 if CPU_FETCH_DRAFT else 4
        print("The model bytes stay on the attached dataset; only packs/ and mtp/ are written locally.")
    else:
        needed_gib = 62 if CPU_FETCH_DRAFT else 56
        print("Plan estimate: about 63.7 GB (58.5 model + 5.2 draft) = ~59.3 GiB;")
        print("the Gyro pack adds little. Keep ~3 GiB of headroom for checkpoints and .part files.")
        if MODEL_VARIANT.lower() == "gyro-m":
            print("\nWARNING: Gyro-M is 92 GB, larger than Kaggle's ~70 GB writable layer.")
            print("Attach it with MODEL_FILES_DATASET and set CPU_DOWNLOAD_MODEL = False.")
    if free_gb < needed_gib:
        print(f"\nWARNING: about {needed_gib} GiB is needed and only {free_gb:.1f} GiB is free.")
        print("Free space first, set CPU_FETCH_DRAFT = False (saves ~4.8 GiB), or point")
        print("MODEL_DATA_DIR at a larger scratch path such as /kaggle/temp/strata-local.")
    print("On Kaggle, /root/strata-local and /kaggle/temp are session scratch outside the")
    print("20 GB /kaggle/working quota; /tmp is not a separate larger disk.")
    if CPU_DOWNLOAD_MODEL and CPU_FETCH_DRAFT and not MODEL_FILES_PATH:
        print("One pass needs ~63 GiB, near Kaggle's ~70 GB writable layer (stay under ~60).")
        print("Use the two-pass flow below unless you have proven more space than that.")
    print("Strata's setup.py is imported directly, so no NVIDIA GPU is required.")
    CPU_SUMMARY = sk.prepare_model_on_cpu(
        MODEL_ROOT, MODEL_FAMILY, MODEL_VARIANT,
        download_model=CPU_DOWNLOAD_MODEL,
        build_pack=CPU_BUILD_PACK,
        fetch_draft=CPU_FETCH_DRAFT,
        model_path=MODEL_FILES_PATH,
    )
    print("\nCPU preparation summary:", json.dumps(CPU_SUMMARY, indent=2))
    if not CPU_SUMMARY.get("pack_ok"):
        print("\nPack not finished on CPU. The GGUF is still on disk: upload the model")
        print("dataset, then let the T4 x2 session build packs/ from it.")
    if not CPU_SUMMARY.get("draft_ok"):
        print("\nDraft layer not finished on CPU. Skip the draft-layer upload; the T4 x2")
        print("session fetches mtp/rt itself.")
else:
    print("Skipped. Set CPU_PREPARE_MODEL = True to download and prepare here.")

print("\nFiles now under", MODEL_DATA_DIR)
sk.cache_status(MODEL_ROOT)
print("\nIf models/ has real bytes, run section 5 in this same session.")


## 4. GPU gate — only if section 3c did not produce the files

Section 3c should be enough. Use this cell only when it reported a missing
helper, or when you deliberately chose to let Strata's normal launch path create
the files.

Switch the accelerator to **GPU T4 ×2** in the notebook settings. Kaggle restarts
the kernel when the accelerator changes, so rerun sections 1–2 first; section 3's
probe does not need repeating.

This cell refuses to continue on a CPU session instead of failing later during a
10–15 minute compile.


In [ ]:
GPU_REPORT = sk.environment_report(require_cuda=True, path=MODEL_DATA_DIR)
DRAFTING_STATUS = sk.drafting_status(MODEL_ROOT, DRAFTING)
print("Drafting status:", json.dumps(DRAFTING_STATUS, indent=2))

free = GPU_REPORT["disk_free_gb"]
print("Free disk on", GPU_REPORT["disk_path"], ":", free, "GiB")
print("Local disk should hold only the runtime. If models/packs show real bytes below,")
print("an earlier 'copy' restore is still occupying space.")
sk.cache_status(MODEL_ROOT)


## 5. Create or refresh the datasets

Run this in the session where the files exist. Order is intentional: runtime
first (cheap, always useful), then the draft layer, then the model last.

If section 3c prepared the files, this works on a CPU session and costs no GPU
hours. Uploading while an accelerator is attached still bills GPU wall-clock, so
do it before switching to T4 ×2.

Wait until every dataset shows **Ready** on its Kaggle page before ending the
session.

If you are using the two-pass flow from section 3c, upload the model with
`UPLOAD_DRAFT_LAYER = False` in pass 1, then the draft layer with
`UPLOAD_MODEL_CACHE = False` in pass 2.


In [ ]:
# Section 3c may already have adopted the bundle before building the pack.
MODEL_FILES_PATH = globals().get("MODEL_FILES_PATH")
if MODEL_FILES_DATASET.strip():
    _bundle_owner, _, _bundle_slug = MODEL_FILES_DATASET.partition("/")
    MODEL_FILES_PATH = sk.adopt_model_bundle(
        _bundle_owner.strip() or KAGGLE_USERNAME, _bundle_slug.strip(), MODEL_ROOT,
        MODEL_FAMILY, MODEL_VARIANT, restore_mode=MODEL_RESTORE_MODE,
    )
    print("Model files adopted from dataset:", MODEL_FILES_PATH)

RUNTIME_UPLOADED = DRAFT_LAYER_UPLOADED = MODEL_UPLOADED = False

if UPLOAD_RUNTIME_CACHE:
    RUNTIME_UPLOADED = sk.upload_runtime_cache(
        username=KAGGLE_USERNAME, slug=RUNTIME_DATASET_SLUG,
        env=KAGGLE_ENV, force=FORCE_UPDATE_RUNTIME_CACHE,
    )
    print("Runtime cache upload submitted:", RUNTIME_UPLOADED)

if UPLOAD_DRAFT_LAYER:
    DRAFT_LAYER_UPLOADED = sk.upload_draft_layer(
        data_dir=MODEL_ROOT, username=KAGGLE_USERNAME,
        slug=DRAFT_LAYER_DATASET_SLUG, env=KAGGLE_ENV,
        force=FORCE_UPDATE_DRAFT_LAYER,
    )
    print("Draft-layer upload submitted:", DRAFT_LAYER_UPLOADED)

if UPLOAD_MODEL_CACHE:
    # An attached bundle is already a dataset. Uploading models/ again would
    # copy its 92 GB into another dataset, so the default is packs/ only;
    # set UPLOAD_ADOPTED_MODEL_FILES = True to duplicate the model too.
    _model_include = ("models", "packs") if (not MODEL_FILES_PATH or UPLOAD_ADOPTED_MODEL_FILES) else ("packs",)
    MODEL_UPLOADED = sk.upload_model_cache(
        data_dir=MODEL_ROOT, username=KAGGLE_USERNAME, slug=MODEL_DATASET_SLUG,
        family=MODEL_FAMILY, variant=MODEL_VARIANT, env=KAGGLE_ENV,
        license_confirmed=I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE,
        force=FORCE_UPDATE_MODEL_CACHE, include=_model_include,
    )
    print("Model cache upload submitted:", MODEL_UPLOADED, "from", "/".join(_model_include))

print("\nDatasets:")
for name in (RUNTIME_DATASET_SLUG, DRAFT_LAYER_DATASET_SLUG, MODEL_DATASET_SLUG):
    print(" -", KAGGLE_USERNAME + "/" + name)


## 6. Verify the datasets exist and are yours

This uses the Kaggle CLI that section 1 installed. It prints status for the three
slugs; the model may be absent until section 5 has run to completion.


In [ ]:
for slug in (RUNTIME_DATASET_SLUG, DRAFT_LAYER_DATASET_SLUG, MODEL_DATASET_SLUG):
    print("=" * 20, slug)
    result = subprocess.run(
        ["kaggle", "datasets", "status", KAGGLE_USERNAME + "/" + slug],
        text=True, capture_output=True, env=KAGGLE_ENV, check=False,
    )
    print((result.stdout or result.stderr or "").strip()[:2000])


## 7. Hand off to the launcher

When all three datasets are **Ready**, next session is:

1. Open `Strata_Kaggle_Launcher.ipynb` and switch the accelerator to **T4 ×2**.
2. Attach under **Add Input → Your Datasets**:
   - `strata-runtime-cache`
   - `strata-mtp-rt`
   - `strata-model-<family>-<variant>`
3. Keep the launcher's defaults: `MODEL_RESTORE_MODE = "symlink"`, and
   `DRAFT_LAYER_DATASET = ""` (which resolves to your own `strata-mtp-rt`).
4. Run the launcher cells in order. Expected startup: runtime restore ~1 min,
   model/packs/draft linked rather than copied, then the plan and launch.

Expected local disk: roughly **1.5 GiB**, not ~80 GiB.

Measure the symlink trade-off once with the launcher's optional quality gate. If
decode tokens/s are unacceptable, the middle ground is copying only `models/`
while symlinking packs and the draft layer — but that needs the free disk this
design is avoiding.

## If something goes wrong

| Symptom | Meaning | Fix |
|---|---|---|
| `restore_model_cache` says "not attached" | dataset missing from Add Input | attach it, rerun section 2 |
| restore says "already existed as local copies and were kept" | an earlier `copy` restore left files | delete `MODEL_DATA_DIR/models`, rerun in `symlink` mode |
| plan fails on CPU with a CUDA error | no CPU-only planning in this build | run section 4 on T4 ×2 |
| upload exits without creating a dataset | license flag is False, or no files under `models/`/`packs/` | confirm the license, check the directory |
| the model dataset is larger than Kaggle's free disk | ~80 GiB may not fit next to runtime | keep the model dataset attached and never copy it; do not use `copy` mode |
